# Introduction to Pandas for Scientific Data Analysis
### Computing for Science II — Guest Lecture

---

**Learning Objectives**

By the end of this lecture, you will be able to:

1. Load tabular data into a pandas DataFrame and inspect its structure
2. Select, filter, and slice data using labels and conditions
3. Handle missing data
4. Compute descriptive statistics and grouped summaries
5. Create quick exploratory visualizations
6. Recognize how pandas compares to what you already know in R

---

**Dataset:** Monthly weather observations (temperature, precipitation, wind, snow) from 7 U.S. cities across 2022–2023.  
**File:** `weather_stations.csv` — place it in the same folder as this notebook.

---

### ⏱ Lecture Roadmap (~50 minutes)

| Section | Topic | ~Time |
|---------|-------|-------|
| 0 | Why pandas? Mental models from R | 5 min |
| 1 | Loading data & first look | 8 min |
| 2 | Selecting and filtering | 10 min |
| 3 | Missing data | 5 min |
| 4 | Descriptive statistics | 7 min |
| 5 | GroupBy — the real workhorse | 8 min |
| 6 | Quick visualization | 5 min |
| 7 | Discussion & exercises | 2 min |


---
## Section 0 — Why Pandas? Connecting to What You Know

If you've used R, you already know `data.frame`. Pandas is essentially that idea, brought into Python.

| Concept | R | pandas |
|---------|---|--------|
| Tabular data object | `data.frame` | `DataFrame` |
| Single column / 1D | `vector` | `Series` |
| Read CSV | `read.csv('file.csv')` | `pd.read_csv('file.csv')` |
| Row/column subset | `df[rows, cols]` | `df.loc[rows, cols]` |
| Summary statistics | `summary(df)` | `df.describe()` |
| Group aggregations | `aggregate()` / `dplyr::group_by` | `df.groupby()` |
| Missing values | `NA` | `NaN` / `pd.NA` |

> **Key mental model:** A pandas `DataFrame` is a dictionary of `Series` objects, where every column is a `Series` sharing a common index.

### Why is pandas so popular in science?
- Reads dozens of file formats (CSV, Excel, NetCDF via xarray, SQL, HDF5, Parquet…)
- Handles missing data gracefully — unlike raw NumPy arrays
- Integrates tightly with NumPy, SciPy, Matplotlib, and scikit-learn
- The `groupby` + `agg` pattern replaces many loops you'd otherwise write


---
## Section 1 — Loading Data and First Look

### 1.1 Import and load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Jupyter magic to display plots inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)  # default figure size

# Load the dataset
df = pd.read_csv('weather_stations.csv')

print("Data loaded successfully!")
print(f"Shape: {df.shape}  →  {df.shape[0]} rows × {df.shape[1]} columns")

### 1.2 Inspect the DataFrame

Always start by getting oriented in your data.

In [ ]:
# Show the first few rows
df.head()

In [ ]:
# Column names, data types, and non-null counts in one shot
df.info()

In [ ]:
# Quick statistical summary of all numeric columns
df.describe()

In [ ]:
# What cities (stations) are in the dataset?
print("Unique stations:")
print(df[['station_id', 'station_name', 'state', 'elevation_m']].drop_duplicates().to_string(index=False))

> **📌 Discussion Question 1:**  
> Look at the output of `df.describe()`. Without doing any further calculations, which city do you *think* has the highest mean maximum temperature? The most precipitation? What clues does `describe()` give you — and what does it hide?

---

### 1.3 Accessing individual columns (Series)

A column is a `Series` — a 1D labeled array.

In [ ]:
# Two equivalent ways to access a column
temps = df['temp_max_c']           # dictionary-style (always works)
# temps = df.temp_max_c            # attribute-style (works when name has no spaces)

print(type(temps))
print(temps.head(10))

In [ ]:
# Select multiple columns at once → returns a DataFrame
df[['station_name', 'year', 'month', 'temp_max_c', 'precip_mm']].head()

---
## Section 2 — Selecting and Filtering Data

### 2.1 Row selection by position and label

Pandas has two main indexers:
- **`.iloc`** — integer position (like Python list indexing)
- **`.loc`** — label-based indexing (uses the DataFrame's index)

In [ ]:
# .iloc: rows 0–4, first 5 columns (exclusive end, like Python slicing)
df.iloc[0:5, 0:5]

In [ ]:
# .loc: select by row index label AND column name (inclusive end!)
df.loc[0:4, 'station_name':'temp_max_c']

> **⚠️ Common gotcha:** `.iloc` slicing is exclusive of the end index (like Python). `.loc` slicing is *inclusive* of the end label. This trips up R users who are used to inclusive indexing everywhere.

### 2.2 Boolean filtering — the bread and butter of data analysis

In [ ]:
# Create a boolean mask
is_summer = (df['month'] >= 6) & (df['month'] <= 8)

# Apply the mask
summer_data = df[is_summer]
print(f"Summer months (June–Aug): {len(summer_data)} rows")
summer_data.head()

In [ ]:
# Find all months where Phoenix exceeded 40°C
extreme_heat = df[(df['station_name'] == 'Phoenix') & (df['temp_max_c'] > 40)]
extreme_heat[['station_name', 'year', 'month', 'temp_max_c']]

In [ ]:
# isin() — filter by a list of values (replaces multiple OR conditions)
west_coast = df[df['station_name'].isin(['Seattle', 'Denver', 'Phoenix'])]
print(f"West/mountain cities: {west_coast['station_name'].unique()}")
west_coast.head()

### 2.3 Adding computed columns

New columns are created by assignment. Operations broadcast across the whole column automatically.

In [ ]:
# Compute temperature range and convert to Fahrenheit
df['temp_range_c'] = df['temp_max_c'] - df['temp_min_c']
df['temp_max_f'] = df['temp_max_c'] * 9/5 + 32

df[['station_name', 'month', 'temp_max_c', 'temp_min_c', 'temp_range_c', 'temp_max_f']].head(6)

> **📌 Discussion Question 2:**  
> Looking at `temp_range_c` (the difference between daily high and low), which city do you predict will have the *largest* diurnal temperature range on average, and why? Think about geography, elevation, humidity, and proximity to water. We'll verify your predictions with `groupby` in Section 5.

---
## Section 3 — Handling Missing Data

Real scientific datasets are almost never complete. Pandas represents missing values as `NaN` (Not a Number) and has built-in tools for finding, filling, or dropping them.

### 3.1 Introducing missing values

Let's simulate the kind of data gap you'd see in practice — a sensor that went offline.

In [ ]:
# Simulate a sensor outage: randomly blank out some wind speed readings
np.random.seed(42)
missing_idx = np.random.choice(df.index, size=20, replace=False)
df_missing = df.copy()                      # always work on a copy!
df_missing.loc[missing_idx, 'wind_speed_mps'] = np.nan

print("Missing wind speed readings:", df_missing['wind_speed_mps'].isna().sum())

### 3.2 Detecting and counting missing values

In [ ]:
# Count missing values per column
print("Missing values per column:")
print(df_missing.isna().sum())

In [ ]:
# As a percentage
pct_missing = df_missing.isna().sum() / len(df_missing) * 100
print("\nPercent missing:")
print(pct_missing[pct_missing > 0].round(1))

### 3.3 Options for dealing with missing data

In [ ]:
# Option 1: Drop rows with any NaN  (often too aggressive)
df_dropped = df_missing.dropna()
print(f"Original rows: {len(df_missing)}   After dropna: {len(df_dropped)}")

# Option 2: Fill with a constant
df_filled_zero = df_missing['wind_speed_mps'].fillna(0)

# Option 3: Fill with the column mean (simple imputation)
mean_wind = df_missing['wind_speed_mps'].mean()
df_filled_mean = df_missing['wind_speed_mps'].fillna(mean_wind)

# Option 4: Forward-fill (carry last valid observation forward — common in time series)
df_filled_ffill = df_missing['wind_speed_mps'].fillna(method='ffill')

print(f"\nColumn mean used for imputation: {mean_wind:.2f} m/s")
print("\nFirst 5 imputed values (mean fill):")
print(df_filled_mean.iloc[missing_idx[:5]])

> **📌 Discussion Question 3:**  
> You have wind speed data with ~12% missing values from a sensor outage. What are the scientific consequences of each approach above?  
> - Dropping rows entirely  
> - Filling with the column mean  
> - Forward-fill (using the previous month's reading)  
>
> Is there a "right" answer? What would you need to know about *why* the data is missing before deciding?

---
## Section 4 — Descriptive Statistics

Pandas makes it easy to compute common statistical summaries directly on Series or DataFrames.

### 4.1 Basic statistics on a Series

In [ ]:
temp_col = df['temp_max_c']

print(f"Count  : {temp_col.count()}")
print(f"Mean   : {temp_col.mean():.2f} °C")
print(f"Median : {temp_col.median():.2f} °C")
print(f"Std Dev: {temp_col.std():.2f} °C")
print(f"Min    : {temp_col.min():.2f} °C")
print(f"Max    : {temp_col.max():.2f} °C")
print(f"Skewness: {temp_col.skew():.3f}")

In [ ]:
# Correlation matrix for numeric variables
numeric_cols = ['temp_max_c', 'temp_min_c', 'precip_mm', 'humidity_pct', 
                'wind_speed_mps', 'snow_depth_cm', 'elevation_m']

corr_matrix = df[numeric_cols].corr()
corr_matrix.round(2)

### 4.2 Sorting and ranking

In [ ]:
# Which 5 months had the highest rainfall across all stations?
top_rain = (df[['station_name', 'year', 'month', 'precip_mm']]
              .sort_values('precip_mm', ascending=False)
              .head(5))
top_rain

In [ ]:
# Sort by multiple columns: station, then year, then month
df_sorted = df.sort_values(['station_name', 'year', 'month'])
df_sorted[['station_name', 'year', 'month', 'temp_max_c']].head(8)

---
## Section 5 — GroupBy: The Real Workhorse

The **split → apply → combine** pattern is how you answer most scientific questions about grouped data:

1. **Split** the data into groups (e.g., by station, by month)
2. **Apply** a function to each group (e.g., mean, sum, custom function)
3. **Combine** the results back into a new DataFrame

This is equivalent to `aggregate()` or `dplyr::group_by() %>% summarize()` in R.

### 5.1 Basic groupby + aggregation

In [ ]:
# Annual mean maximum temperature by city
annual_temp = (df.groupby(['station_name', 'year'])['temp_max_c']
                 .mean()
                 .round(2)
                 .reset_index())
annual_temp

In [ ]:
# Multiple statistics at once using .agg()
city_climate = df.groupby('station_name').agg(
    mean_tmax   = ('temp_max_c',  'mean'),
    mean_tmin   = ('temp_min_c',  'mean'),
    total_precip= ('precip_mm',   'sum'),
    max_snow    = ('snow_depth_cm','max'),
    mean_wind   = ('wind_speed_mps','mean')
).round(2)

city_climate

> **📌 Discussion Question 4:**  
> Look at `city_climate` above. Miami has very high precipitation but very low wind speeds, while Chicago has high wind speeds but moderate precipitation. How does the physical geography of each location explain these patterns? Can you identify any surprising results in this table?

In [ ]:
# Seasonal cycle: average temperature by month across all cities
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

seasonal_cycle = (df.groupby('month')
                    .agg(mean_tmax=('temp_max_c','mean'),
                         mean_tmin=('temp_min_c','mean'),
                         mean_precip=('precip_mm','mean'))
                    .round(2))

seasonal_cycle.index = seasonal_cycle.index.map(month_names)
seasonal_cycle

In [ ]:
# Verify our earlier prediction: which city has the largest daily temp range?
range_by_city = (df.groupby('station_name')['temp_range_c']
                   .mean()
                   .sort_values(ascending=False)
                   .round(2))

print("Mean daily temperature range (°C) by city:")
print(range_by_city)

### 5.2 Using a custom function with groupby

You're not limited to built-in aggregation functions. You can pass any function.

In [ ]:
# Count how many months per city exceeded 30°C
def count_hot_months(series):
    return (series > 30).sum()

hot_months = (df.groupby('station_name')['temp_max_c']
                .agg(count_hot_months)
                .sort_values(ascending=False)
                .rename('months_above_30c'))

print("Months with max temp > 30°C (across 2022–2023):")
print(hot_months)

In [ ]:
# Using transform() to add a group-level statistic BACK to the original DataFrame
# This is useful for anomaly detection!
df['station_mean_tmax'] = df.groupby('station_name')['temp_max_c'].transform('mean')
df['tmax_anomaly'] = df['temp_max_c'] - df['station_mean_tmax']

# What months were most anomalously warm?
top_anomalies = (df[['station_name', 'year', 'month', 'temp_max_c', 'tmax_anomaly']]
                   .sort_values('tmax_anomaly', ascending=False)
                   .head(8))
top_anomalies

> **🎯 Mini-Exercise (2 min):**  
> Using what you've learned, write a single `groupby().agg()` call to find the **total annual precipitation** for each city in each year. Which city-year combination was wettest? Which was driest?

*(Hint: group by `['station_name', 'year']`, aggregate `precip_mm` with `'sum'`)*

In [ ]:
# Your code here!


---
## Section 6 — Quick Visualizations

Pandas has a built-in `.plot()` interface that wraps Matplotlib. It's great for fast exploratory plots — not publication-ready figures, but enough to see patterns quickly.

### 6.1 Seasonal temperature cycle

In [ ]:
# Build a pivot table: rows = month, columns = city, values = mean max temp
pivot_temp = (df.groupby(['station_name', 'month'])['temp_max_c']
                .mean()
                .unstack(level=0))  # cities become columns

# Plot all cities on one figure
ax = pivot_temp.plot(marker='o', linewidth=2)
ax.set_title('Mean Monthly Maximum Temperature by City (2022–2023)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (°C)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.legend(title='City', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 6.2 Precipitation comparison — bar chart

In [ ]:
# Mean monthly precipitation by city — horizontal bar chart
mean_precip = df.groupby('station_name')['precip_mm'].mean().sort_values()

ax = mean_precip.plot(kind='barh', color='steelblue', edgecolor='white')
ax.set_title('Mean Monthly Precipitation by City', fontsize=13)
ax.set_xlabel('Precipitation (mm)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 6.3 Scatter plot — temperature vs. humidity

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

cities = df['station_name'].unique()
colors = plt.cm.tab10.colors

for i, city in enumerate(sorted(cities)):
    subset = df[df['station_name'] == city]
    ax.scatter(subset['temp_max_c'], subset['humidity_pct'],
               label=city, color=colors[i], alpha=0.7, s=50)

ax.set_title('Maximum Temperature vs. Relative Humidity', fontsize=13)
ax.set_xlabel('Max Temperature (°C)')
ax.set_ylabel('Relative Humidity (%)')
ax.legend(title='City', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

> **📌 Discussion Question 5:**  
> Look at the scatter plot. There appears to be a negative relationship between temperature and humidity for most cities — as temperature goes up, humidity tends to go down. But Miami seems to break this pattern. Why might that be? What does this tell you about the limits of analyzing all cities together without considering regional climate context?

### 6.4 Histogram of temperature values

In [ ]:
df['temp_max_c'].plot(kind='hist', bins=25, edgecolor='black', color='tomato', alpha=0.8)
plt.axvline(df['temp_max_c'].mean(), color='navy', linestyle='--', linewidth=2, label=f"Mean: {df['temp_max_c'].mean():.1f}°C")
plt.axvline(df['temp_max_c'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {df['temp_max_c'].median():.1f}°C")
plt.title('Distribution of Monthly Maximum Temperatures (All Cities)', fontsize=13)
plt.xlabel('Max Temperature (°C)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 7 — Pulling It Together: A Mini-Analysis

Let's answer a realistic science question from start to finish:

**Question:** *Is there a relationship between a city's elevation and the size of its seasonal temperature range?*

In [ ]:
# Step 1: Compute annual temperature range for each city (max - min across all months)
city_stats = df.groupby('station_name').agg(
    elevation_m    = ('elevation_m',   'first'),
    annual_tmax    = ('temp_max_c',    'max'),
    annual_tmin    = ('temp_min_c',    'min'),
    mean_temp_range = ('temp_range_c', 'mean')
).reset_index()

city_stats['annual_range'] = city_stats['annual_tmax'] - city_stats['annual_tmin']
city_stats.sort_values('elevation_m')

In [ ]:
# Step 2: Visualize the relationship
from scipy import stats

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(city_stats['elevation_m'], city_stats['annual_range'],
           s=100, color='steelblue', zorder=5)

# Annotate each point with city name
for _, row in city_stats.iterrows():
    ax.annotate(row['station_name'],
                xy=(row['elevation_m'], row['annual_range']),
                xytext=(8, 2), textcoords='offset points', fontsize=9)

# Add a linear regression line
slope, intercept, r, p, se = stats.linregress(city_stats['elevation_m'],
                                               city_stats['annual_range'])
x_line = np.linspace(city_stats['elevation_m'].min(), city_stats['elevation_m'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=1.5,
        label=f'Linear fit  (r={r:.2f}, p={p:.3f})')

ax.set_title('Elevation vs. Annual Temperature Range', fontsize=13)
ax.set_xlabel('Elevation (m)')
ax.set_ylabel('Annual Temperature Range (°C)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nLinear regression: slope = {slope:.4f} °C per meter of elevation")
print(f"R² = {r**2:.3f}   p-value = {p:.4f}")

---
## 📝 Discussion Questions (Summary)

Use these for class discussion or as short-answer reflection questions:

1. **`describe()` and hidden structure:** `df.describe()` gives you a statistical summary of the entire dataset. What information does it *hide* by lumping all cities together? How might you get more meaningful summaries?

2. **Predicting diurnal temperature range:** Before running any code, which city would you predict has the largest average daily temperature range? What physical factors (elevation, proximity to water, humidity, cloud cover) govern diurnal temperature range? Did the data confirm your prediction?

3. **Missing data decisions:** Imagine 20 wind speed readings are missing because a sensor failed on particularly windy days (not at random). How would this bias each of the imputation strategies — dropping rows, mean fill, and forward fill? Is there a strategy that would be least biased in this scenario?

4. **GroupBy interpretation:** The `city_climate` table shows Miami has the highest total precipitation and Phoenix has the lowest. What physical mechanisms explain this? Can you think of a month or season where this pattern might *reverse*?

5. **Humidity and temperature scatter:** The scatter plot shows a generally negative correlation between temperature and humidity — except Miami breaks the pattern. What is it about Miami's climate (or climate type) that creates this exception? What does this say about using global statistics vs. stratified analysis?

6. **Correlation is not causation:** The correlation matrix showed that `temp_max_c` and `humidity_pct` are negatively correlated across our dataset. Does this mean higher temperatures *cause* lower humidity? What confounding variables (like location, season, or air mass type) might explain this correlation?

7. **Pandas vs. R (for R users):** Which pandas operation felt most analogous to something you already do in R? Which felt most different or counterintuitive? What would you want to explore further?

---
## 🔧 Take-Home Exercises

These are short, self-contained challenges to practice what you learned:

**Exercise 1:** Create a new column called `feels_hot` that is `True` when `temp_max_c > 35` AND `humidity_pct > 60` (a rough proxy for oppressive heat). Which city-month combinations satisfy this condition?

**Exercise 2:** Using `groupby`, compute the **coefficient of variation** (CV = std / mean × 100) of monthly precipitation for each city. CV measures relative variability. Which city has the most *variable* rainfall month-to-month? What might explain this?

**Exercise 3:** Build a plot showing the monthly snow depth for Denver, Chicago, and Boston on the same axes. Which city has the most persistent snow cover? Does snow depth peak in the same month as the coldest temperatures?

**Exercise 4 (challenge):** Calculate a simple **aridity index** for each city as `mean_annual_temp_max / total_annual_precip`. Rank the cities from most arid to least arid. Does this ranking match your physical intuition?

---
## 📚 What's Next?

Pandas has much more to explore:

- **`pd.merge()` and `pd.concat()`** — joining multiple DataFrames (like SQL joins or R's `merge()`)
- **Time series indexing** — `pd.to_datetime()`, `resample()`, rolling windows
- **`pd.pivot_table()`** — cross-tabulations and pivot summaries
- **`pd.melt()` / `pd.wide_to_long()`** — reshaping between wide and long formats
- **Seaborn** — a statistical visualization library built on Matplotlib that works beautifully with pandas DataFrames
- **Xarray** — pandas-like tool for multi-dimensional scientific arrays (gridded climate data, NetCDF files)

**Resources:**
- [Pandas documentation](https://pandas.pydata.org/docs/) — excellent and searchable
- [Python for Data Analysis (McKinney)](https://wesmckinney.com/book/) — free online, written by the creator of pandas
- [Pandas cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
